# Robot de tri médical — entraînement des yeux

**Partie A** : détecteur RF-DETR (trouve *où* sont les objets), jeu Roboflow « Hospital Consumables ».
**Partie B** : classifieur (dit *ce que c'est* sur l'objet découpé), images images.cv.

Avant de commencer :
1. **Exécution → Modifier le type d'exécution → GPU T4**
2. 🔑 **Secrets** (barre de gauche) : `ROBOFLOW_API_KEY` = votre clé Roboflow (Settings → API Keys), accès au notebook activé.
3. Lancer les cellules **dans l'ordre**. Tout ce qui compte est enregistré dans Google Drive (`MyDrive/modeles/` et `MyDrive/datasets/`).

## A1. Carte graphique, paquets, Google Drive

In [ ]:
!nvidia-smi -L
!pip install -q "rfdetr[train,loggers]" roboflow onnx onnxsim supervision
# Si Colab propose « Redémarrer la session » après l'installation : cliquer, puis relancer cette cellule.

from google.colab import drive
drive.mount("/content/drive")
SORTIE_DET = "/content/drive/MyDrive/modeles/rfdetr_medical"
SORTIE_CLS = "/content/drive/MyDrive/modeles/classifieur_medical"
import os
os.makedirs(SORTIE_DET, exist_ok=True)
os.makedirs(SORTIE_CLS, exist_ok=True)

## A2. Télécharger le jeu Roboflow (version 1, format COCO)

In [ ]:
from roboflow import Roboflow
from google.colab import userdata
import json, pathlib

rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
projet = rf.workspace("star-developer-6073").project("hospital-consumables-7zdur")
jeu = projet.version(1).download("coco")

for split in ["train", "valid", "test"]:
    f = pathlib.Path(jeu.location, split, "_annotations.coco.json")
    if f.exists():
        d = json.loads(f.read_text())
        print(split, len(d["images"]), "images")
CATEGORIES_DET = {c["id"]: c["name"] for c in d["categories"]}
print("Classes :", CATEGORIES_DET)

## A3. Entraîner RF-DETR Nano (15 à 40 min — laisser l'onglet ouvert)
En cas de `CUDA out of memory` : `batch_size=2, grad_accum_steps=8`.

In [ ]:
from rfdetr import RFDETRNano

modele = RFDETRNano()
modele.train(
    dataset_dir=jeu.location,
    epochs=50,
    batch_size=4,
    grad_accum_steps=4,
    output_dir=SORTIE_DET,
)

## A4. Tester sur les images de test

In [ ]:
import glob
import supervision as sv
from PIL import Image

!ls {SORTIE_DET}
poids = sorted(glob.glob(f"{SORTIE_DET}/checkpoint_best*.pth"))
print("Poids utilisés :", poids[0])
meilleur = RFDETRNano(pretrain_weights=poids[0])

for img_path in sorted(glob.glob(f"{jeu.location}/test/*.jpg"))[:3]:
    img = Image.open(img_path)
    det = meilleur.predict(img, threshold=0.5)
    etiquettes = [f"{CATEGORIES_DET.get(int(c), c)} {s:.2f}" for c, s in zip(det.class_id, det.confidence)]
    vue = sv.BoxAnnotator().annotate(img.copy(), det)
    vue = sv.LabelAnnotator().annotate(vue, det, etiquettes)
    sv.plot_image(vue)

## A5. Exporter le détecteur en ONNX (fichier qui tourne hors ligne sur le Mac)

In [ ]:
meilleur.export(output_dir=SORTIE_DET)
json.dump(CATEGORIES_DET, open(f"{SORTIE_DET}/classes.json", "w"), ensure_ascii=False, indent=1)
!ls -la {SORTIE_DET}

## B1. images.cv : télécharger les catégories médicales (autonome)
Les zips vont dans Drive (`MyDrive/datasets/imagescv/`) : après une déconnexion, relancer cette cellule ne retélécharge rien.
Licence images.cv / ImageNet : **recherche seulement** — à montrer à Hiba.

In [ ]:
import requests, secrets, time, zipfile, shutil
from pathlib import Path

API = "https://api.images.cv"
ENTETES = {
    "Content-Type": "application/json", "Accept": "application/json",
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
    "Origin": "https://images.cv", "Referer": "https://images.cv/",
}
CATEGORIES_CLS = ["syringe", "bandage", "band-aid", "scalpel", "clamp", "straight-scissor",
                  "curved-scissor", "pill-bottle", "stethoscope", "mask", "neck-brace", "medicine-chest"]
ZIPS = Path("/content/drive/MyDrive/datasets/imagescv")
FUSION = Path("/content/icv_fusion")
ZIPS.mkdir(parents=True, exist_ok=True)

def appel(chemin, corps):
    r = requests.post(API + chemin, json=corps, headers=ENTETES, timeout=60)
    if "application/json" not in r.headers.get("Content-Type", ""):
        raise RuntimeError(f"{chemin} : réponse non JSON ({r.status_code}) : {r.text[:200]!r}")
    d = r.json()
    if r.status_code != 200 or d.get("isError"):
        raise RuntimeError(f"{chemin} : erreur {r.status_code} : {d}")
    return d["data"]

def nombre_images(nom):
    return appel("/get_items_image_category", {"category": nom, "start_after": 0}).get("sumCounter") or 0

def telecharger(nom, dest):
    corps = {"output_size": "size_256", "color_type": "color", "images": [], "categories": [nom],
             "augmentations": [], "split_set": [0.7, 0.15, 0.15], "link_id": secrets.token_hex(20),
             "request_type": "CALL_FROM_SEARCH", "user_email": [], "user_id": "", "augmentation_percent": 30}
    lien_id = appel("/download_images", corps)["link_id"]
    fin = time.time() + 1200
    while time.time() < fin:
        url = appel("/get_link", {"link_id": lien_id}).get("public_url")
        if url:
            with requests.get(url, stream=True, timeout=120) as r, open(dest, "wb") as f:
                for morceau in r.iter_content(1 << 20):
                    f.write(morceau)
            with open(dest, "rb") as f:
                if f.read(2) != b"PK":
                    dest.unlink()
                    raise RuntimeError("le fichier reçu n'est pas un zip")
            return
        time.sleep(5)
    raise RuntimeError("trop long : le zip n'a pas été prêt en 20 min")

bilan = {}
for slug in CATEGORIES_CLS:
    zip_path = ZIPS / f"{slug}.zip"
    if not (zip_path.exists() and zip_path.stat().st_size > 10_000):
        nom = next((s for s in [slug, slug.replace("-", "_")] if nombre_images(s)), None)
        if not nom:
            print(f"✗ {slug} : introuvable")
            continue
        for essai in range(3):
            try:
                print(f"↓ {slug}…")
                telecharger(nom, zip_path)
                break
            except Exception as e:
                print(f"   essai {essai + 1} raté : {str(e)[:400]}")
                time.sleep(60 * (essai + 1))
        else:
            print(f"✗ {slug} : abandonné pour l'instant (relancer plus tard)")
            continue
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            parties = Path(m).parts
            if "data" in parties and not m.endswith("/"):
                i = parties.index("data")
                if len(parties) >= i + 4 and parties[i + 1] in ("train", "val", "test"):
                    dst = FUSION / parties[i + 1] / slug / parties[-1]
                    dst.parent.mkdir(parents=True, exist_ok=True)
                    with z.open(m) as src, open(dst, "wb") as out:
                        shutil.copyfileobj(src, out)
    bilan[slug] = {s: len(list((FUSION / s / slug).glob("*"))) for s in ("train", "val", "test")}
    print(f"✓ {slug} : {bilan[slug]}")

print("\nBILAN :", bilan)

## B2. Regarder les images de chaque classe
ImageNet : `clamp` peut être une pince de menuisier, `mask` un masque de carnaval… Pour retirer une classe : `!rm -rf /content/icv_fusion/*/clamp`

In [ ]:
import random
import matplotlib.pyplot as plt
from PIL import Image

classes = sorted(p for p in (FUSION / "train").iterdir() if any(p.iterdir()))
fig, axes = plt.subplots(len(classes), 6, figsize=(12, 2 * len(classes)))
for ligne, c in zip(axes, classes):
    fichiers = list(c.iterdir())
    for ax, p in zip(ligne, random.sample(fichiers, min(6, len(fichiers)))):
        ax.imshow(Image.open(p))
    for ax in ligne:
        ax.axis("off")
    ligne[0].set_title(c.name, loc="left", fontsize=9)
plt.tight_layout()
plt.show()

## B3. Entraîner le classifieur (MobileNetV3) et l'exporter en ONNX

In [ ]:
import json
import torch, torch.nn as nn
from torchvision import datasets, transforms, models

dev = "cuda" if torch.cuda.is_available() else "cpu"
norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
tf_train = transforms.Compose([transforms.RandomResizedCrop(224, scale=(0.6, 1)), transforms.RandomHorizontalFlip(),
                               transforms.ColorJitter(0.3, 0.3, 0.3), transforms.ToTensor(), norm])
tf_eval = transforms.Compose([transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(), norm])
tr = datasets.ImageFolder(FUSION / "train", tf_train)
va = datasets.ImageFolder(FUSION / "val", tf_eval)
dl_tr = torch.utils.data.DataLoader(tr, batch_size=64, shuffle=True, num_workers=2)
dl_va = torch.utils.data.DataLoader(va, batch_size=64, num_workers=2)
print("Classes :", tr.classes, "|", len(tr), "images d'entraînement")

net = models.mobilenet_v3_large(weights="IMAGENET1K_V2")
net.classifier[3] = nn.Linear(net.classifier[3].in_features, len(tr.classes))
net = net.to(dev)
opt = torch.optim.AdamW(net.parameters(), lr=3e-4)
for epoch in range(8):
    net.train()
    for x, y in dl_tr:
        opt.zero_grad()
        nn.functional.cross_entropy(net(x.to(dev)), y.to(dev)).backward()
        opt.step()
    net.eval(); ok = tot = 0
    with torch.no_grad():
        for x, y in dl_va:
            ok += (net(x.to(dev)).argmax(1).cpu() == y).sum().item(); tot += len(y)
    print(f"passe {epoch + 1} : précision validation {ok / tot:.1%}")

torch.onnx.export(net.cpu().eval(), torch.randn(1, 3, 224, 224), f"{SORTIE_CLS}/classifieur.onnx",
                  input_names=["image"], output_names=["scores"], dynamic_axes={"image": {0: "n"}})
json.dump(tr.classes, open(f"{SORTIE_CLS}/classes.json", "w"))
print("Exporté dans", SORTIE_CLS)